# Superposition scaling reproduction

Reproduces Figures 3 and 4. The plotting cells load from `data/*.npz` and save to `figures/`.

Training and export cells need the model checkpoints (trained on Colab, not included) and are behind `%%script echo` toggles; delete that line to run them.

## Setup

In [ ]:
import math, os, glob, time, itertools
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.optimizer import Optimizer
import matplotlib.pyplot as plt
from matplotlib import colors, cm
from matplotlib.lines import Line2D
from scipy.stats import linregress

In [ ]:
# checkpoints: only used by the skipped training/export cells (originally run on Colab)
CKPT_ROOT = '/content/drive/MyDrive/superposition_repro'
FIG3A_DIR = f'{CKPT_ROOT}/fig3a'
FIG3B_DIR = f'{CKPT_ROOT}/fig3b'
FIG4_SWEEP_DIR = f'{CKPT_ROOT}/fig4_sweep'

# exported arrays and figures, relative to notebooks/
DATA_DIR = '../data'
FIG_DIR = '../figures'
os.makedirs(FIG_DIR, exist_ok=True)

n = 1000
m_values = [10, 15, 25, 39, 63, 100]
gamma_values = np.linspace(-1, 1, 10)
alphas = np.linspace(0, 2, 17)

### Optimizer, data, model, training

In [ ]:
# Adapted from liuyz0/SuperpositionScaling, exp/adamw.py

class AdamW(Optimizer):
    def __init__(
        self,
        params,
        lr=1e-3,
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=1e-2,
        amsgrad=False,
    ):
        """
        AdamW implementation with logistic weight decay

        Args:
            params (iterable): Iterable of parameters to optimize
            lr (float, optional): Learning rate (default: 1e-3)
            betas (tuple[float, float], optional): Coefficients for computing running averages of gradient and its square (default: (0.9, 0.999))
            eps (float, optional): Term added to denominator to improve numerical stability (default: 1e-8)
            weight_decay (float, optional): Weight decay coefficient (default: 1e-2)
            amsgrad (bool, optional): Whether to use the AMSGrad variant (default: False)
        """
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
        if not 0.0 <= eps:
            raise ValueError(f"Invalid epsilon value: {eps}")
        if not 0.0 <= betas[0] < 1.0:
            raise ValueError(f"Invalid beta parameter at index 0: {betas[0]}")
        if not 0.0 <= betas[1] < 1.0:
            raise ValueError(f"Invalid beta parameter at index 1: {betas[1]}")
        #if not 0.0 <= weight_decay:
            #raise ValueError(f"Invalid weight_decay value: {weight_decay}")

        defaults = dict(
            lr=lr,
            betas=betas,
            eps=eps,
            weight_decay=weight_decay,
            amsgrad=amsgrad
        )
        super().__init__(params, defaults)

    def __setstate__(self, state):
        super().__setstate__(state)
        for group in self.param_groups:
            group.setdefault('amsgrad', False)

    @torch.no_grad()
    def step(self, closure=None):
        """
        Performs a single optimization step.

        Args:
            closure (callable, optional): A closure that reevaluates the model and returns the loss
        """
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            params_with_grad = []
            grads = []
            exp_avgs = []
            exp_avg_sqs = []
            max_exp_avg_sqs = []
            state_steps = []
            beta1, beta2 = group['betas']

            for p in group['params']:
                if p.grad is None:
                    continue
                params_with_grad.append(p)
                if p.grad.is_sparse:
                    raise RuntimeError('AdamW does not support sparse gradients')
                grads.append(p.grad)

                state = self.state[p]

                # State initialization
                if len(state) == 0:
                    state['step'] = 0
                    # Exponential moving average of gradient values
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    # Exponential moving average of squared gradient values
                    state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    if group['amsgrad']:
                        # Maintains max of all exp. moving avg. of sq. grad. values
                        state['max_exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format)

                exp_avgs.append(state['exp_avg'])
                exp_avg_sqs.append(state['exp_avg_sq'])

                if group['amsgrad']:
                    max_exp_avg_sqs.append(state['max_exp_avg_sq'])

                state['step'] += 1
                state_steps.append(state['step'])

            # Perform stepweight decay
            for p in params_with_grad:
                if group['weight_decay'] >= 0:
                    p.mul_(1 - group['weight_decay'] * group['lr'])
                else:
                    row_norms = p.norm(dim=1).unsqueeze(1).add_(group['eps'])
                    decay_factor = group['weight_decay'] * p * (1 - 1/row_norms)
                    p.add_(decay_factor, alpha=group['lr'])

            # Update momentum and variance
            for i, param in enumerate(params_with_grad):
                grad = grads[i]
                exp_avg = exp_avgs[i]
                exp_avg_sq = exp_avg_sqs[i]
                step = state_steps[i]

                bias_correction1 = 1 - beta1 ** step
                bias_correction2 = 1 - beta2 ** step

                # Update biased first moment estimate
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                # Update biased second raw moment estimate
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                if group['amsgrad']:
                    max_exp_avg_sq = max_exp_avg_sqs[i]
                    # Maintains the maximum of all 2nd moment running avg. till now
                    torch.maximum(max_exp_avg_sq, exp_avg_sq, out=max_exp_avg_sq)
                    # Use the max. for normalizing running avg. of gradient
                    denom = (max_exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(group['eps'])
                else:
                    denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(group['eps'])

                step_size = group['lr'] / bias_correction1
                param.addcdiv_(exp_avg, denom, value=-step_size)

        return loss

In [ ]:
# power-law probability distribution
def feature_probs(n, alpha, E = 1.0, device = None):
  i = torch.arange(1, n+1, dtype=torch.float32, device=device)
  p = i ** (-alpha)
  return E * p / p.sum()

# samples a batch with bernoulli drawing
def sampler(p, batch_size = 2048):
  u = p.expand(batch_size, -1)
  u = torch.bernoulli(u)
  v = torch.rand([batch_size, p.numel()], device = p.device)*2
  x = u * v
  return x

In [ ]:
class ToyModel(nn.Module):
  def __init__(self, n, m):
    super().__init__()
    self.W = nn.Parameter(torch.randn(n, m) / math.sqrt(m))
    self.b = nn.Parameter(torch.randn(n))


  def forward(self, x):
    h = x @ self.W
    y = torch.relu(h @ self.W.T + self.b)
    return y

In [ ]:
# Linear warmup, then cosine decay to min_lr (schedule from the paper's text).
# All runs below pass warmup_ratio=0.1, i.e. 2000 of 20000 steps.

def lr_multiplier(step, n_steps, warmup_ratio = 0.05):
  warmup_steps = int(warmup_ratio*n_steps)
  step += 1
  min_lr = 0.05
  if step < warmup_steps:
    return step/warmup_steps
  else:
    # taken from paper text
    return (1.0 - min_lr) * 0.5 * (1 + math.cos(math.pi * (step - warmup_steps) / (n_steps - warmup_steps))) + min_lr

In [ ]:
def training_loop(n, m, alpha, batch_size, n_steps, warmup_ratio, gamma, base_lr = 1e-2, E= 1.0, seed = 0, device = "cuda"):
    torch.manual_seed(seed)
    p = feature_probs(n, alpha, E, device)
    model = ToyModel(n, m).to(device)


    optimizer = AdamW([
          {"params": [model.W], "lr": base_lr, "weight_decay": gamma},
          {"params": [model.b], "lr": base_lr, "weight_decay": 0.0},
      ])

    for g in optimizer.param_groups:
      g['init_lr'] = g['lr']

    losses = torch.empty(n_steps, device = device)

    for step in range(n_steps):
      x = sampler(p, batch_size) # b x n
      y = model(x) # b x n
      loss = ((y-x)**2).mean()

      optimizer.zero_grad()
      loss.backward()
      losses[step] = loss.detach()

      for g in optimizer.param_groups:
        g['lr'] = g['init_lr'] * lr_multiplier(step, n_steps, warmup_ratio)

      optimizer.step()

    with torch.no_grad():
        test_loss = sum(
            ((model(xt) - xt) ** 2).sum(dim=1).mean().item()
            for xt in (sampler(p, batch_size) for _ in range(100))
        ) / 100

    return {
        "W": model.W.detach().cpu(),
        "b": model.b.detach().cpu(),
        "losses": losses.cpu(),
        "test_loss": test_loss,
        "cfg": dict(n=n, m=m, alpha=alpha, E=E, gamma=gamma, seed=seed,
                    batch_size=batch_size, n_steps=n_steps,
                    base_lr=base_lr, warmup_ratio=warmup_ratio),
    }

In [ ]:
def cached_run(path, **cfg):
    """Load a saved run if it exists, otherwise train, save, and return it."""
    if os.path.exists(path):
        saved = torch.load(path, weights_only=False)
        assert saved['config'] == cfg, f"{path} was trained with {saved['config']}"
        print(f"loaded {path}")
        return saved['out']
    out = training_loop(**cfg)
    out = {k: v.detach().cpu() if torch.is_tensor(v) else v for k, v in out.items()}
    torch.save({'config': cfg, 'out': out}, path)
    print(f"saved {path}")
    return out


def run_path(a, g, m):
    return f"{FIG4_SWEEP_DIR}/a{a:.3f}_g{g:+.3f}_m{m}.pt"


def save_fig(fig, name):
    for ext in ("png", "pdf"):
        path = f"{FIG_DIR}/{name}.{ext}"
        fig.savefig(path, dpi=200, bbox_inches="tight")
    print("saved", name)

## Figure 3

### Figure 3a

In [ ]:
%%script echo "skip 3a training cell"

runs, norms = {}, {}
for gamma in [1.0, -1.0]:
    path = f"{FIG3A_DIR}/gamma{gamma:+.0f}.pt"
    runs[gamma] = cached_run(path, n=n, m=100, alpha=1, batch_size=2048,
                             n_steps=20000, warmup_ratio=0.1, gamma=gamma)
    norms[gamma] = runs[gamma]['W'].norm(dim=1).numpy()
    print(f"γ={gamma:+.0f}  φ½={(norms[gamma] > 0.5).mean():.3f}  "
          f"test loss={runs[gamma]['test_loss']:.3e}")

In [ ]:
%%script echo "skip 3a export cell"

# export: checkpoints -> fig3a_norms.npz
gammas_3a = [1.0, -1.0]
saved = {g: torch.load(f"{FIG3A_DIR}/gamma{g:+.0f}.pt", weights_only=False) for g in gammas_3a}
cfg = saved[1.0]["config"]
np.savez(f"{DATA_DIR}/fig3a_norms.npz",
         gammas=np.array(gammas_3a),
         norms=np.stack([saved[g]["out"]["W"].norm(dim=1).numpy() for g in gammas_3a]),
         n=cfg["n"], alpha=cfg["alpha"])
print("saved fig3a_norms.npz")

In [ ]:
d = np.load(f"{DATA_DIR}/fig3a_norms.npz")
norms = {float(g): w for g, w in zip(d["gammas"], d["norms"])}
n = int(d["n"])

p = feature_probs(n, float(d["alpha"])).cpu().numpy()
rank = np.arange(1, n + 1)
bins = np.linspace(0, max(w.max() for w in norms.values()) * 1.02, 26)
reds, unit = plt.get_cmap("Reds"), colors.Normalize(0, 1)

fig, axes = plt.subplots(2, 2, figsize=(5.5, 4), constrained_layout=True)
for col, (gamma, title) in enumerate([(1.0, "Weight decay +"), (-1.0, "Weight decay −")]):
    w = norms[gamma]
    axes[0, col].hist(w, bins=bins, density=True)
    axes[0, col].set(yscale="log", xlabel="Row norm", title=title)
    axes[1, col].bar(rank, p, width=1.0, color=reds(unit(np.clip(w, 0, 1))), linewidth=0)
    axes[1, col].set(xscale="log", yscale="log", xlabel="Rank")
axes[0, 0].set_ylabel("Density")
axes[1, 0].set_ylabel("Freq.")
fig.colorbar(cm.ScalarMappable(unit, reds), ax=axes[1, 1])

save_fig(fig, "fig3a")
plt.show()

### Figure 3b

In [ ]:
%%script echo "skip 3b training cell"

norms_3b = np.full((len(m_values), len(gamma_values), n), np.nan, dtype=np.float32)

for i, m in enumerate(m_values):
    for j, gamma in enumerate(gamma_values):
        t0 = time.time()
        path = f"{FIG3B_DIR}/m{int(m)}_gamma{float(gamma):+.2f}.pt"
        out = cached_run(path, n=n, m=int(m), alpha=1, batch_size=2048, n_steps=20000,
                         warmup_ratio=0.1, gamma=float(gamma))
        norms_3b[i, j] = out["W"].norm(dim=1).numpy()
        print(f"m={m:3d}  γ={gamma:+.2f}  φ½={(norms_3b[i, j] > 0.5).mean():.3f}  ({time.time() - t0:.0f}s)")

In [ ]:
%%script echo "skip 3b export cell"

# export: checkpoints -> fig3b_norms.npz
def ckpt_path(m, gamma):
    return f"{FIG3B_DIR}/m{int(m)}_gamma{float(gamma):+.2f}.pt"

missing = [ckpt_path(m, g) for m in m_values for g in gamma_values
           if not os.path.exists(ckpt_path(m, g))]
assert not missing, f"{len(missing)} checkpoints missing, e.g. {missing[:3]}"

norms = np.full((len(m_values), len(gamma_values), n), np.nan, dtype=np.float32)
for i, m in enumerate(m_values):
    for j, gamma in enumerate(gamma_values):
        ckpt = torch.load(ckpt_path(m, gamma), map_location="cpu", weights_only=False)
        cfg, W = ckpt["config"], ckpt["out"]["W"]
        assert W.shape == (n, m), (m, gamma, tuple(W.shape))
        assert cfg["n"] == n and cfg["m"] == m and cfg["alpha"] == 1, (m, gamma, cfg)
        assert abs(cfg["gamma"] - gamma) < 1e-6, (m, gamma, cfg["gamma"])
        norms[i, j] = W.norm(dim=1).numpy()

np.savez(f"{DATA_DIR}/fig3b_norms.npz", norms=norms,
         m_values=np.array(m_values), gamma_values=gamma_values, n=n, alpha=1.0)
print("saved", norms.shape)

In [ ]:
d = np.load(f"{DATA_DIR}/fig3b_norms.npz")
norms_3b, m_vals, gammas, n = d["norms"], d["m_values"], d["gamma_values"], int(d["n"])
assert norms_3b.shape == (len(m_vals), len(gammas), n), norms_3b.shape
assert not np.isnan(norms_3b).any(), "some runs are missing (NaN entries)"

phi = (norms_3b > 0.5).mean(axis=-1)   # (len(m_vals), len(gammas))
civ, lognorm = plt.get_cmap("cividis"), colors.Normalize(1, 2)

fig, ax = plt.subplots(figsize=(4.5, 3.2), constrained_layout=True)
for i, m in enumerate(m_vals):
    c = civ(lognorm(np.log10(m)))
    ax.hlines(m / n, gammas[0], gammas[-1], colors=[c], linestyles="--")
    ax.scatter(gammas, phi[i], color=c, s=40, zorder=3)
ax.set(xlabel=r"Weight decay, $\gamma$", ylabel=r"Rep. fraction, $\phi_{1/2}$")
ax.legend(handles=[Line2D([], [], marker="o", ls="", color="k", label="Data"),
                   Line2D([], [], ls="--", color="k", label=r"$m/n$")], frameon=False)
fig.colorbar(cm.ScalarMappable(lognorm, civ), ax=ax, label=r"$\log_{10} m$")

save_fig(fig, "fig3b")
plt.show()

## Figure 4

In [ ]:
%%script echo "skip figure 4 training cell"

configs = list(itertools.product(alphas, gamma_values, m_values))
t0 = time.time()
for i, (a, g, m) in enumerate(configs, 1):
    path = run_path(a, g, m)
    if os.path.exists(path):
        continue                                          # finished in an earlier session
    r = training_loop(1000, m, float(a), 2048, 20000, 0.1, float(g))
    r["k"] = (r["W"].norm(dim=1) > 0.5).sum().item()
    r = {key: (v.cpu() if torch.is_tensor(v) else v) for key, v in r.items()}  # loadable without a GPU
    torch.save(r, path + ".tmp")
    os.replace(path + ".tmp", path)                       # rename only after a complete write
    print(f"[{i}/{len(configs)}] a={a:.3f} g={g:+.3f} m={m}  "
          f"k={r['k']}  loss={r['test_loss']:.4f}  ({time.time() - t0:.0f}s)")

n_saved = sum(os.path.exists(run_path(*c)) for c in configs)
print(f"done: {n_saved}/{len(configs)} runs saved")

In [ ]:
%%script echo "skip figure 4 load cell"

# load all Figure 4 runs (used by both exports below)
runs = [torch.load(f, weights_only=False) for f in sorted(glob.glob(f"{FIG4_SWEEP_DIR}/*.pt"))]
print(len(runs), "runs loaded")

### Figure 4a

In [ ]:
def theory_loss(k, alpha, n=1000):
    phi = k / n
    if alpha == 1:
        return (4/3) * -np.log(phi) / np.log(n)
    return (4/3) * (phi**(1 - alpha) - 1) / (1 - n**(1 - alpha)) * n**(1 - alpha)

In [ ]:
%%script echo "skip 4a export cell"

# export: runs -> fig4a_runs.npz (one entry per run)
np.savez(f"{DATA_DIR}/fig4a_runs.npz",
         k=np.array([r["k"] for r in runs]),
         test_loss=np.array([r["test_loss"] for r in runs]),
         alpha=np.array([r["cfg"]["alpha"] for r in runs]),
         gamma=np.array([r["cfg"]["gamma"] for r in runs]),
         m=np.array([r["cfg"]["m"] for r in runs]))
print("saved fig4a_runs.npz")

In [ ]:
d = np.load(f"{DATA_DIR}/fig4a_runs.npz")
pos = d["gamma"] > 0                     # weak-superposition runs, as in Fig. 4a
k, loss, alpha = d["k"][pos], d["test_loss"][pos], d["alpha"][pos]

fig = plt.figure()
plt.scatter(k, loss, c=alpha, cmap="plasma", vmin=0, vmax=2, s=14)
plt.colorbar(label="Data exponent, α")

ks = np.logspace(0, np.log10(700), 200)
for a in [0, 0.5, 1, 1.5, 2]:
    plt.plot(ks, theory_loss(ks, a), "--", color=plt.cm.plasma(a / 2))

plt.xscale("log"); plt.yscale("log")
plt.xlabel("Number of represented features"); plt.ylabel("Loss")
save_fig(fig, "fig4a")
plt.show()

### Figure 4b

In [ ]:
# grid-index lookups for (m, gamma, alpha)
M_IDX = {m: i for i, m in enumerate(m_values)}

def m_to_idx(m):
    return M_IDX[int(m)]

def linspace_inverse(grid, atol=1e-6):
    start, step, num = grid[0], grid[1] - grid[0], len(grid)
    def to_idx(value):
        i = int(round((value - start) / step))
        if not (0 <= i < num) or abs(grid[i] - value) > atol:
            raise ValueError(f"{value} is not on the grid")
        return i
    return to_idx

gamma_to_idx = linspace_inverse(gamma_values)
alpha_to_idx = linspace_inverse(alphas)

In [ ]:
%%script echo "skip 4b ambiguity cell"

def ambiguity_norm(m, n, W):
  reference = torch.zeros([n, n])
  reference[:m , :m] = torch.eye(m)
  return torch.linalg.matrix_norm(W @ W.T - reference, ord = "fro")

ambiguity = torch.full([len(m_values), len(gamma_values), len(alphas)], float("nan"))

for r in runs:
  m, gamma, alpha = r['cfg']['m'], r['cfg']['gamma'], r['cfg']['alpha']
  m_i = m_to_idx(m)
  g_i = gamma_to_idx(gamma)
  a_i = alpha_to_idx(alpha)
  n = r['cfg']['n']
  w = r["W"]
  ambiguity[m_i, g_i, a_i] = ambiguity_norm(m,n,w)

In [ ]:
%%script echo "skip 4b case selection cell"

# pick the 9 (gamma, alpha) cases with the smallest worst-case ambiguity over m
worst = ambiguity.flatten(1, 2).max(dim = 0).values
points = torch.topk(worst, k=9, largest = False)
g_i, alpha_i = np.unravel_index(points.indices.numpy(), ambiguity.shape[1:])

In [ ]:
%%script echo "skip 4b fit cell"

case_alpha = alphas[alpha_i]
case_gamma = gamma_values[g_i]

losses = []
for id in range(9):
  curr_losses = []
  alpha, gamma = case_alpha[id], case_gamma[id]
  for m in m_values:
    ckpt = torch.load(run_path(alpha, gamma, m), weights_only=False)
    curr_losses.append(ckpt["test_loss"])
  losses.append(curr_losses)

fits = [linregress(np.log(m_values), np.log(losses[c])) for c in range(9)]
alpha_m = np.array([-f.slope for f in fits])
alpha_m_error = np.array([f.stderr for f in fits])

for c in range(9):
    print(f"α={case_alpha[c]:.3f}  γ={case_gamma[c]:+.2f}  α_m={alpha_m[c]:.2f} ± {alpha_m_error[c]:.2f}")

In [ ]:
%%script echo "skip 4b export cell"

# export: ambiguity grid and fits -> fig4_frobenius.npz
np.savez(
    f"{DATA_DIR}/fig4_frobenius.npz",
    ambiguity=ambiguity.numpy(),          # (6, 10, 17): m × γ × α
    m_values=np.array(m_values),
    gamma_values=gamma_values,
    alphas=alphas,
    g_i=g_i, alpha_i=alpha_i,             # the 9 picks, as grid positions
    worst=points.values.numpy(),          # their worst-case ambiguity
    alpha_m=alpha_m, alpha_m_error=alpha_m_error,
    norm="fro",                           # records which norm produced this
)
print("saved fig4_frobenius.npz")

In [ ]:
d = np.load(f"{DATA_DIR}/fig4_frobenius.npz")
case_alpha = d["alphas"][d["alpha_i"]]
case_gamma = d["gamma_values"][d["g_i"]]
alpha_m, alpha_m_error = d["alpha_m"], d["alpha_m_error"]

cmap, norm = plt.get_cmap("Spectral"), colors.Normalize(-1, 1)

fig, ax = plt.subplots(figsize=(4, 3), constrained_layout=True)
for c in range(9):
    ax.errorbar(case_alpha[c], alpha_m[c], yerr=alpha_m_error[c], fmt="o",
                color=cmap(norm(case_gamma[c])), ms=7)
ax.plot([1, 2], [0, 1], "k--", label=r"$\alpha_m = \alpha - 1$")
ax.set(xlabel=r"Data exponent, $\alpha$", ylabel=r"Model exponent, $\alpha_m$",
       xlim=(1, 2.05), ylim=(0, 1.3))
ax.legend(frameon=False)
fig.colorbar(cm.ScalarMappable(norm, cmap), ax=ax, label=r"Weight decay, $\gamma$")

save_fig(fig, "fig4b")
plt.show()